# 05. 원곡 그룹 기반 데이터 분할

`master_manifest.csv`를 `original_audio` 단위로 Train, Validation, Test에 나눈다. 같은 원곡의 REAL과 FAKE가 서로 다른 split에 포함되지 않도록 먼저 원곡별 표를 만든 뒤, 장르 비율을 기준으로 70:15:15에 가깝게 분할한다.

난수 시드는 42로 고정해 같은 분할을 다시 만들 수 있도록 했다.


In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "data").is_dir() and (p / "results").is_dir()),
    Path.cwd().resolve(),
)

MASTER_PATH = PROJECT_ROOT / "data/metadata/master_manifest.csv"
GROUP_SPLIT_PATH = PROJECT_ROOT / "data/metadata/original_audio_split.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/metadata/master_manifest_with_split.csv"

RANDOM_STATE = 42

print("MASTER_PATH      :", MASTER_PATH)
print("GROUP_SPLIT_PATH :", GROUP_SPLIT_PATH)
print("OUTPUT_PATH      :", OUTPUT_PATH)
print("RANDOM_STATE     :", RANDOM_STATE)


MASTER_PATH      : <PROJECT_ROOT>/data/metadata/master_manifest.csv
GROUP_SPLIT_PATH : <PROJECT_ROOT>/data/metadata/original_audio_split.csv
OUTPUT_PATH      : <PROJECT_ROOT>/data/metadata/master_manifest_with_split.csv
RANDOM_STATE     : 42


**실행 결과.** 입력 manifest와 두 출력 CSV의 경로를 지정하고 `RANDOM_STATE`를 42로 설정했다. 결과 파일은 `data/metadata/` 아래에 저장한다.


## 1. Master manifest 확인

통합 manifest를 불러와 전체 행 수, 원곡 수, 라벨 및 장르 분포를 확인한다.


In [2]:
master = pd.read_csv(MASTER_PATH)

print("===== MASTER MANIFEST =====")
print("Rows                 :", len(master))
print("Unique original_audio:", master["original_audio"].nunique())

print("\nLabel distribution:")
print(master["label"].value_counts())

print("\nGenre distribution:")
print(master["genre"].value_counts())

display(master.head())


===== MASTER MANIFEST =====
Rows                 : 3458
Unique original_audio: 296

Label distribution:
label
FAKE    3162
REAL     296
Name: count, dtype: int64

Genre distribution:
genre
Electronic    1249
Rock          1238
Pop            971
Name: count, dtype: int64


,sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,description,path_in_dataset,file_exists
0,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,NaN,NaN,True
1,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,NaN,NaN,True
2,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,NaN,NaN,True
3,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/114/114244.mp3,114244.0,NaN,NaN,True
4,sample_00004,3 am West End - statusq,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/112/112378.mp3,112378.0,NaN,NaN,True


**실행 결과.** 입력 데이터는 3,458행, 원곡 296개다. 라벨은 FAKE 3,162개와 REAL 296개이며, 장르는 Electronic 1,249개, Rock 1,238개, Pop 971개다.


## 2. 원곡 단위 표 생성

`original_audio`별 장르와 REAL·FAKE 수를 집계한다. 한 원곡 안에서 장르가 달라지거나 REAL이 하나가 아닌 경우가 있는지도 검사한다.


In [3]:
group_table = (
    master
    .groupby("original_audio")
    .agg(
        genre=("genre", "first"),
        genre_count=("genre", "nunique"),
        total_samples=("sample_id", "size"),
        real_count=("label", lambda x: (x == "REAL").sum()),
        fake_count=("label", lambda x: (x == "FAKE").sum()),
    )
    .reset_index()
)

print("Groups:", len(group_table))
print("Groups with genre_count != 1:", int((group_table["genre_count"] != 1).sum()))
print("Groups with real_count != 1 :", int((group_table["real_count"] != 1).sum()))

print("\nGroup-level genre distribution:")
print(group_table["genre"].value_counts())

display(group_table.head())


Groups: 296
Groups with genre_count != 1: 0
Groups with real_count != 1 : 0

Group-level genre distribution:
genre
Electronic    118
Rock          111
Pop            67
Name: count, dtype: int64


,original_audio,genre,genre_count,total_samples,real_count,fake_count
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,1,6,1,5
1,1984 - Punk Rock Opera,Rock,1,14,1,13
2,2 (Wasn't There) - Isle of Pine,Rock,1,17,1,16
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,1,6,1,5
4,3 am West End - statusq,Electronic,1,6,1,5


**실행 결과.** 원곡 단위 표는 296행이다. 장르가 둘 이상인 그룹과 REAL이 하나가 아닌 그룹은 모두 0개였고, 원곡 장르는 Electronic 118개, Rock 111개, Pop 67개로 집계되었다.


## 3. Train / Validation / Test 분할

원곡 그룹을 먼저 Train 70%와 임시 집합 30%로 나누고, 임시 집합을 Validation과 Test로 다시 나눈다. 두 단계 모두 `genre`를 기준으로 층화 추출한다.


In [4]:
train_groups, temp_groups = train_test_split(
    group_table,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=group_table["genre"],
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_groups["genre"],
)

train_groups = train_groups.copy()
val_groups = val_groups.copy()
test_groups = test_groups.copy()

train_groups["split"] = "train"
val_groups["split"] = "val"
test_groups["split"] = "test"

group_split = pd.concat(
    [train_groups, val_groups, test_groups],
    ignore_index=True,
)

print("===== GROUP SPLIT COUNTS =====")
print(group_split["split"].value_counts())

print("\nTotal groups:", len(group_split))


===== GROUP SPLIT COUNTS =====
split
train    207
test      45
val       44
Name: count, dtype: int64

Total groups: 296


**실행 결과.** 원곡 그룹은 Train 207개, Validation 44개, Test 45개로 배정되었다. 전체 296개 가운데 각각 69.93%, 14.86%, 15.20%에 해당한다.


## 4. 그룹 단위 장르 분포

split별 원곡 수와 장르 비율을 비교해 층화 추출 결과를 확인한다.


In [5]:
group_genre_count = pd.crosstab(
    group_split["split"],
    group_split["genre"],
)

group_genre_ratio = pd.crosstab(
    group_split["split"],
    group_split["genre"],
    normalize="index",
).round(4)

print("===== GROUP-LEVEL GENRE COUNT =====")
display(group_genre_count)

print("===== GROUP-LEVEL GENRE RATIO =====")
display(group_genre_ratio)


===== GROUP-LEVEL GENRE COUNT =====


genre,Electronic,Pop,Rock
split,,,
test,18,10,17
train,82,47,78
val,18,10,16


===== GROUP-LEVEL GENRE RATIO =====


genre,Electronic,Pop,Rock
split,,,
test,0.4000,0.2222,0.3778
train,0.3961,0.2271,0.3768
val,0.4091,0.2273,0.3636


**실행 결과.** 원곡 기준 장르 비율은 Train이 39.61% / 22.71% / 37.68%, Validation이 40.91% / 22.73% / 36.36%, Test가 40.00% / 22.22% / 37.78%(Electronic / Pop / Rock)였다.


## 5. Sample 단위 split 부여

원곡별 split 정보를 3,458개 sample에 결합한다. 같은 `original_audio`를 가진 행은 모두 같은 split 값을 받는다.


In [6]:
split_map = group_split[["original_audio", "split"]].copy()

master_split = master.merge(
    split_map,
    on="original_audio",
    how="left",
    validate="many_to_one",
)

print("Rows:", len(master_split))
print("Missing split:", master_split["split"].isna().sum())

print("\nSample-level split distribution:")
print(master_split["split"].value_counts())

display(master_split.head())


Rows: 3458
Missing split: 0

Sample-level split distribution:
split
train    2392
test      539
val       527
Name: count, dtype: int64


,sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,description,path_in_dataset,file_exists,split
0,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,NaN,NaN,True,train
1,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,NaN,NaN,True,train
2,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,NaN,NaN,True,val
3,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/114/114244.mp3,114244.0,NaN,NaN,True,test
4,sample_00004,3 am West End - statusq,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/112/112378.mp3,112378.0,NaN,NaN,True,test


**실행 결과.** 3,458개 sample 모두에 split이 부여되었고 결측치는 없었다. Sample 수는 Train 2,392개, Validation 527개, Test 539개다.


## 6. 원곡 중복 검사

Train, Validation, Test의 `original_audio` 집합을 두 개씩 비교해 split 사이에 같은 원곡이 겹치는지 확인한다.


In [7]:
train_ids = set(group_split.loc[group_split["split"] == "train", "original_audio"])
val_ids = set(group_split.loc[group_split["split"] == "val", "original_audio"])
test_ids = set(group_split.loc[group_split["split"] == "test", "original_audio"])

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("===== ORIGINAL_AUDIO OVERLAP CHECK =====")
print("Train ∩ Val :", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Val ∩ Test  :", len(val_test_overlap))

all_group_ids = train_ids | val_ids | test_ids

print("\nAssigned original_audio:", len(all_group_ids))
print("Expected original_audio:", master["original_audio"].nunique())


===== ORIGINAL_AUDIO OVERLAP CHECK =====
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Assigned original_audio: 296
Expected original_audio: 296


**실행 결과.** Train–Validation, Train–Test, Validation–Test 사이의 원곡 중복은 모두 0개였다. 배정된 원곡 수도 예상값과 같은 296개였다.


## 7. Split별 REAL / FAKE 분포

원곡별로 REAL이 하나씩 있으므로 각 split의 REAL 수는 원곡 그룹 수와 같다. FAKE 수와 라벨 비율도 함께 확인한다.


In [8]:
label_count = pd.crosstab(
    master_split["split"],
    master_split["label"],
)

print("===== LABEL DISTRIBUTION BY SPLIT =====")
display(label_count)

label_ratio = pd.crosstab(
    master_split["split"],
    master_split["label"],
    normalize="index",
).round(4)

print("===== LABEL RATIO BY SPLIT =====")
display(label_ratio)


===== LABEL DISTRIBUTION BY SPLIT =====


label,FAKE,REAL
split,,
test,494,45
train,2185,207
val,483,44


===== LABEL RATIO BY SPLIT =====


label,FAKE,REAL
split,,
test,0.9165,0.0835
train,0.9135,0.0865
val,0.9165,0.0835


**실행 결과.** Train은 FAKE 2,185개와 REAL 207개, Validation은 483개와 44개, Test는 494개와 45개로 구성되었다. FAKE 비율은 세 split에서 91.35~91.65%였다.


## 8. Split별 장르 분포

원곡마다 FAKE 생성물 수가 달라질 수 있으므로 sample 단위 장르 비율을 별도로 계산한다.


In [9]:
sample_genre_count = pd.crosstab(
    master_split["split"],
    master_split["genre"],
)

sample_genre_ratio = pd.crosstab(
    master_split["split"],
    master_split["genre"],
    normalize="index",
).round(4)

print("===== SAMPLE-LEVEL GENRE COUNT =====")
display(sample_genre_count)

print("===== SAMPLE-LEVEL GENRE RATIO =====")
display(sample_genre_ratio)


===== SAMPLE-LEVEL GENRE COUNT =====


genre,Electronic,Pop,Rock
split,,,
test,185,141,213
train,894,677,821
val,170,153,204


===== SAMPLE-LEVEL GENRE RATIO =====


genre,Electronic,Pop,Rock
split,,,
test,0.3432,0.2616,0.3952
train,0.3737,0.2830,0.3432
val,0.3226,0.2903,0.3871


**실행 결과.** Sample 기준 장르 비율은 Train 37.37% / 28.30% / 34.32%, Validation 32.26% / 29.03% / 38.71%, Test 34.32% / 26.16% / 39.52%(Electronic / Pop / Rock)였다.


## 9. Split별 생성기 분포

FAKE만 선택해 12개 생성기의 Train, Validation, Test 배정 수를 확인한다.


In [10]:
fake_only = master_split[master_split["label"] == "FAKE"].copy()

generator_count = pd.crosstab(
    fake_only["split"],
    fake_only["generator"],
)

print("===== GENERATOR DISTRIBUTION BY SPLIT =====")
display(generator_count)


===== GENERATOR DISTRIBUTION BY SPLIT =====


generator,acestep,audioldm,brev,diffrhythm,elevenlabs,mubert,musicgen,producer,songgen,stableaudio,suno,udio
split,,,,,,,,,,,,
test,45,45,48,47,48,24,45,25,45,26,48,48
train,206,203,204,207,204,100,205,102,205,141,204,204
val,43,44,46,45,48,25,43,24,42,27,48,48


**실행 결과.** 12개 생성기가 모두 세 split에 포함되었다. 예를 들어 `acestep`은 Train 206개, Validation 43개, Test 45개였고, `mubert`는 각각 100개, 25개, 24개였다.


## 10. 최종 품질 검사

원곡 296개의 배정 여부, split 간 중복, sample 수, 결측 split, REAL·FAKE 수를 종합해 확인한다.


In [11]:
qc_summary = pd.DataFrame({
    "check": [
        "total_groups",
        "train_groups",
        "val_groups",
        "test_groups",
        "train_val_overlap",
        "train_test_overlap",
        "val_test_overlap",
        "master_rows",
        "missing_split",
        "real_rows",
        "fake_rows",
    ],
    "value": [
        len(all_group_ids),
        len(train_ids),
        len(val_ids),
        len(test_ids),
        len(train_val_overlap),
        len(train_test_overlap),
        len(val_test_overlap),
        len(master_split),
        int(master_split["split"].isna().sum()),
        int((master_split["label"] == "REAL").sum()),
        int((master_split["label"] == "FAKE").sum()),
    ],
})

display(qc_summary)

core_qc_pass = (
    len(all_group_ids) == 296
    and len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
    and len(master_split) == 3458
    and int(master_split["split"].isna().sum()) == 0
    and int((master_split["label"] == "REAL").sum()) == 296
    and int((master_split["label"] == "FAKE").sum()) == 3162
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", core_qc_pass)


,check,value
0,total_groups,296
1,train_groups,207
2,val_groups,44
3,test_groups,45
4,train_val_overlap,0
5,train_test_overlap,0
6,val_test_overlap,0
7,master_rows,3458
8,missing_split,0
9,real_rows,296


===== FINAL RESULT =====
Core QC PASS: True


**실행 결과.** 원곡 296개와 sample 3,458개가 모두 배정되었고 split 중복 및 결측은 0개였다. REAL 296개와 FAKE 3,162개도 유지되어 `Core QC PASS`가 `True`였다.


## 11. 결과 저장

원곡 단위 배정표와 sample 단위 manifest를 각각 CSV로 저장한다.


In [12]:
if not core_qc_pass:
    raise RuntimeError(
        "Split Core QC가 통과하지 않았습니다. 저장 전에 위 결과를 확인하세요."
    )

GROUP_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

group_split[
    [
        "original_audio",
        "genre",
        "total_samples",
        "real_count",
        "fake_count",
        "split",
    ]
].sort_values(["split", "original_audio"]).to_csv(
    GROUP_SPLIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

master_split.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved group split :", GROUP_SPLIT_PATH)
print("Saved master split:", OUTPUT_PATH)
print("Rows              :", len(master_split))


Saved group split : <PROJECT_ROOT>/data/metadata/original_audio_split.csv
Saved master split: <PROJECT_ROOT>/data/metadata/master_manifest_with_split.csv
Rows              : 3458


**실행 결과.** 원곡 296개의 배정표를 `original_audio_split.csv`에, split이 추가된 3,458행을 `master_manifest_with_split.csv`에 저장했다.


## 12. 정리

원곡 296개는 Train 207개, Validation 44개, Test 45개로 나뉘었다. 이에 따라 sample은 Train 2,392개, Validation 527개, Test 539개로 배정되었다. split 사이에 겹치는 원곡과 split 결측치는 없었다.


## 다음 단계

이후 EDA와 모델 실험에서는 `master_manifest_with_split.csv`의 split을 그대로 사용한다. 원곡 단위 분할을 다시 수행하지 않아 실험 사이의 데이터 구성을 동일하게 유지한다.
